# 🏨 Expedia Hotel Ranking Project
This notebook presents data preprocessing, exploration, and modeling for predicting hotel booking preferences using various machine learning methods including LightGBM, KNN, SVD, and Balanced Random Forest.

## 📚 Import Libraries

In [ ]:
# 🔧 General-purpose packages
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

# 📊 Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# 📏 Preprocessing and data handling
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, cross_val_score
from imblearn.over_sampling import SMOTE

# 🤖 Models and evaluation
import lightgbm as lgb
from sklearn.neighbors import KNeighborsClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ndcg_score
from scipy.sparse.linalg import svds

# 🔍 Hyperparameter tuning
import optuna

## 📂 Load Data

In [ ]:
# Replace with actual dataset path 
df = pd.read_csv("expedia_data.csv")
df.head()

## 🛠️ Feature Engineering & Composite Features
This section creates the transformed and composite features. 

In [ ]:
df['price_diff'] = df['visitor_hist_adr_usd'] - df['price_usd']
df['star_diff'] = df['visitor_hist_starrating'] - df['prop_starrating']
df['score_ratio'] = df['prop_review_score'] / (df['prop_location_score2'] + 1)
df['price_rank'] = df.groupby('srch_id')['price_usd'].rank()
df['location_score_sum'] = df['prop_location_score1'] + df['prop_location_score2']
df['price_usd_log'] = np.log1p(df['price_usd'])

# Fill inf/-inf resulting from invalid divisions
df.replace([np.inf, -np.inf], np.nan, inplace=True)

## 🔀 Dataset Splitting
Split the dataset into training, validation, and testing sets.

In [ ]:
# First, create X and y
X = df.drop("booking_bool", axis=1)
y = df["booking_bool"]

# Split into training+validation and testing
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Then split training+validation into train and validation
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42)  # 0.25 x 0.8 = 0.2

print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)


## 🧹 Data Cleaning & Preprocessing

### 🔧 Handling Missing Values

In [ ]:
# Impute missing values
# Fill hotel reviews and search log queries with 0
cols_fill_zero = ['prop_review_score', 'prop_location_score2', 'srch_query_affinity_score']
df[cols_fill_zero] = df[cols_fill_zero].fillna(0)

# Fill continuous variables with median, binary/categorical with mode
for col in df.columns:
    if df[col].dtype in ['float64', 'int64']:
        df[col].fillna(df[col].median(), inplace=True)
    else:
        df[col].fillna(df[col].mode()[0], inplace=True)

### 🚫 Outlier Removal

In [ ]:
# Remove outliers using IQR method
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1
df = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

### 📏 Feature Scaling

In [ ]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

###  🔧 Feature-target split

In [ ]:
# Feature-target split
X = df.drop("booking_bool", axis=1)
y = df["booking_bool"]

### 🧬 Balancing Classes with SMOTE

In [ ]:
X = df_scaled.drop("booking_bool", axis=1)
y = df_scaled["booking_bool"]

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

## 📊 Exploratory Data Analysis

### 📉 Missing Data Heatmap

In [ ]:
# Check missing values
missing = df.isnull().sum()
print(missing[missing > 0])

# Visualize missingness
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values Heatmap")
plt.show()

### 🔗 Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, cmap="coolwarm", annot=False, fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

### 📊 Booking Distribution

In [ ]:
sns.countplot(x="booking_bool", data=df)
plt.title("Distribution of Booking vs. Non-Booking")
plt.xlabel("Booking Made")
plt.ylabel("Count")
plt.xticks([0, 1], ["No", "Yes"])
plt.show()

### 💰 Boxplot of Price by Booking

In [ ]:
sns.boxplot(x="booking_bool", y="price_usd", data=df)
plt.title("Price Distribution by Booking Status")
plt.xlabel("Booking Made")
plt.ylabel("Price (USD)")
plt.xticks([0, 1], ["No", "Yes"])
plt.ylim(0, df["price_usd"].quantile(0.95))  # Clip outliers for visibility
plt.show()

## 🧪📊 Model Evaluation with Hyperparameter Tuning & Metrics

### 📂 Load Data

In [ ]:
# Replace with actual path to your data
train = r"/Users/train_dataset.csv"
df_train = pd.read_csv(train)
df_train.head()

#test = r"/Users/test_dataset.csv"
#df_test = pd.read_csv(test)
#df_test.head()

#valid= r"/Users/valid_dataset.csv"
#df_valid = pd.read_csv(valid)
#df_valid.head()

### 🔧 Light Gradient Boosting Machine (LightGBM) 

In [ ]:
def create_submission(model1, model2, Test_data, proba = False):
    import time
    start = time.time()
    
    Test = Test_data
    #predictions
    print("~ Making predictions")
    if proba == True:
        clickpreds = model1.predict_proba(Test.drop(['srch_id','prop_id'], axis = 1))
        bookpreds = model2.predict_proba(Test.drop(['srch_id','prop_id'], axis = 1))
        
        Test['click_preds'] = pd.DataFrame(clickpreds)[1].values
        Test['booking_preds'] = pd.DataFrame(bookpreds)[1].values
    else:
        clickpreds = model1.predict(Test.drop(['srch_id','prop_id'], axis = 1))
        bookpreds = model2.predict(Test.drop(['srch_id','prop_id'], axis = 1))
    
        Test['click_preds'] = clickpreds
        Test['booking_preds'] = bookpreds
    
    #Grab 4 columns from Test
    sub = Test[['srch_id','prop_id','click_preds','booking_preds']]
    #Weight booking column values
    
    if proba == True:
        sub['booking_preds'] = sub['booking_preds'].apply(lambda x: x*2 if x > 0 else x)
    else:
        sub['booking_preds'] = sub['booking_preds'].apply(lambda x: x+1 if x > 0 else x)
    #Total click & booking values, remove columns
    sub['total'] = sub['click_preds']+sub['booking_preds']
    sub.drop(['click_preds', 'booking_preds'], axis = 1, inplace = True)
    
    #Sort chunks of srch ids by 'total' value
    print(f"~ Sorting chunks of srch_ids: {(time.time()-start):.2f}s have passed")
    ids = sub['srch_id'].unique()
    sub1 = pd.DataFrame()
    k=0
    
    srch_id = []
    prop_id = []
    total = []
    for item in ids:
       
        df = sub[sub['srch_id'] == item].sort_values('total', ascending = False)
        srch_id.append(df['srch_id'].values)
        prop_id.append(df['prop_id'].values)
        total.append(df['total'].values)
    
        k+=1
        if not k%10000:
            print(f"{k}/{len(ids)} srch_ids, time elapsed: {(time.time() - start):.2f}s")

    #Unpack list of arrays
    print(f"~ Unpacking lists: {(time.time() - start):.2f}s elapsed")
    _srch_id = []
    for i in range(0,len(srch_id)):
        for k in range(0,len(srch_id[i])):
            _srch_id.append(srch_id[i][k])
    _prop_id = []
    for i in range(0,len(prop_id)):
        for k in range(0,len(prop_id[i])):
            _prop_id.append(prop_id[i][k])
    _total = []
    for i in range(0,len(total)):
        for k in range(0,len(total[i])):
            _total.append(total[i][k])
    
    #Create Dataframe from unpacked lists
    print(f"~ Creating Dataframe: {(time.time() - start):.2f}s elapsed")
    sub2 = pd.DataFrame(zip(_srch_id, _prop_id, _total),
               columns =['srch_id', 'prop_id','total'])
    
    sub1 = sub2[['srch_id','prop_id']]
    
    print(f"~ Finished. Total time: {(time.time() - start):.2f}")
    return(sub1, sub2)

In [ ]:
X_train = df_train.drop(['srch_id', 'prop_id','click_bool', 'booking_bool'], axis = 1)
Y1_train = df_train['click_bool']
Y2_train = df_train['booking_bool']

In [ ]:
LGBMclick = lgb.LGBMClassifier(scale_pos_weight = 12, n_estimators = 76)
LGBMbook = lgb.LGBMClassifier(scale_pos_weight = 12, n_estimators = 78)

In [ ]:
LGBMclick.fit(X_train, Y1_train)
LGBMbook.fit(X_train, Y2_train)

### 📌 Feature Importance: LightGBM

In [ ]:
sorted_idx = LGBMclick.feature_importances_.argsort()
plt.barh(X_train.columns[sorted_idx], LGBMclick.feature_importances_[sorted_idx])
plt.xlabel("LightGBM Click Feature Importance")
plt.title("LightGBM Click Model: Features by Importance")

In [ ]:
sorted_idx = LGBMbook.feature_importances_.argsort()
plt.barh(X_train.columns[sorted_idx], LGBMbook.feature_importances_[sorted_idx])
plt.xlabel("Importance")
plt.title("LightGBM Booking Model: Features by Importance")

###  📂 Create File for Kaggle Competition Submission

In [ ]:
results = create_submission(LGBMclick,LGBMbook,df_test, proba = True)

### 📍 K-Nearest Neighbors (KNN)

In [ ]:
# Define objective function for Optuna
def objective(trial):
    n_neighbors = trial.suggest_int("n_neighbors", 3, 25)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    leaf_size = trial.suggest_int("leaf_size", 10, 50)

    knn = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights=weights,
        leaf_size=leaf_size
    )
    
    # Use 3-fold cross-validation with F1 as metric
    score = cross_val_score(knn, X_train, y_train, cv=3, scoring="f1").mean()
    return score

# Create and run Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Train best model
best_params = study.best_params
knn_best = KNeighborsClassifier(**best_params)
knn_best.fit(X_train, y_train)

# Make predictions and evaluate
y_pred_knn = knn_best.predict(X_test)

print("✅ KNN Best Parameters from Optuna:", best_params)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))
print("NDCG Score:", ndcg_score([y_test], [y_pred_knn]))

### 📉 Confusion Matrix: KNN

In [ ]:
cm = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Booking', 'Booking'], yticklabels=['No Booking', 'Booking'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - KNN')
plt.show()

### 🌲 Balanced Random Forest (BRF)

In [ ]:
# Define Optuna objective function
def objective_brf(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 300)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)

    brf = BalancedRandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1
    )
    
    score = cross_val_score(brf, X_train, y_train, scoring='f1', cv=3).mean()
    return score

# Run Optuna optimization
study_brf = optuna.create_study(direction="maximize")
study_brf.optimize(objective_brf, n_trials=50)

# Train best BRF model from Optuna results
brf_best = BalancedRandomForestClassifier(**study_brf.best_params, random_state=42)
brf_best.fit(X_train, y_train)

# Make predictions
y_pred_brf = brf_best.predict(X_test)

# Evaluation
print("BRF Best Parameters from Optuna:", study_brf.best_params)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_brf))
print("NDCG Score:", ndcg_score([y_test], [y_pred_brf]))

### 📉 Confusion Matrix: Balanced Random Forest

In [ ]:
cm = confusion_matrix(y_test, y_pred_brf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Booking', 'Booking'], yticklabels=['No Booking', 'Booking'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - BRF')
plt.show()

### 🔍 Singular Value Decomposition (SVD)

In [ ]:
# Optuna tuning
def objective(trial):
    params = {
        'n_factors': trial.suggest_int('n_factors', 10, 50),
        'n_epochs': trial.suggest_int('n_epochs', 10, 50),
        'lr_all': trial.suggest_float('lr_all', 0.002, 0.05),
        'reg_all': trial.suggest_float('reg_all', 0.01, 0.1)
    }
    model = SVD(**params)
    model.fit(trainset)
    predictions = model.test(testset)
    
    y_true = [int(true_r >= 0.5) for (_, _, true_r, _, _) in predictions]
    y_pred = [int(est >= 0.5) for (_, _, _, est, _) in predictions]
    return ndcg_score([y_true], [y_pred])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# Train with best params
best_params = study.best_params
svd_best = SVD(**best_params)
svd_best.fit(trainset)
predictions = svd_best.test(testset)

# Evaluation
y_test = [int(true_r >= 0.5) for (_, _, true_r, _, _) in predictions]
y_pred_svd = [int(est >= 0.5) for (_, _, _, est, _) in predictions]

print("SVD Evaluation:")
print(classification_report(y_test, y_pred_svd))
print("NDCG Score:", ndcg_score([y_test], [y_pred_svd]))

### 📉 Confusion Matrix: SVD 

In [ ]:
cm = confusion_matrix(y_test, y_pred_svd)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Booking', 'Booking'], yticklabels=['No Booking', 'Booking'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - SVD')
plt.show()

## 🎯 Custom NDCG@5 Evaluation Function

In [ ]:
#Takes in:
#- df_preds, your predictions in the format required for submission to kaggle
#- df_actual, a dataframe that contains target bool values AND srch_ids 
#             (ie Test set that contains X and y)

def NDCG(df_preds, df_actual):
    start = time.time()
    IDCG_per_id = []
    DCG_5_per_id = []
    
    ids = df_preds['srch_id'].unique()
    
    # For each user, score each of the user's query
    score_index = -1
    
    #### For printing TIME #####
    ids_length = len(ids)
    t = 0
    start = time.time()
    ############################
    
    for item in ids: 
        t+=1
        if not t%1000:
            print(f"{t}/{ids_length} ids passed: Time elapsed: {time.time() - start}")
        
        ####Printing time #####
        #t+=1
        #if t%100 == 0:
        #    print(f" ID {t}/{ids_length}, time elapsed: {time.time() - start}")
        #######################
        
        score_index += 1
        IDCG_per_id.append(0)
        DCG_5_per_id.append(0)
        
        # chunk of submission dataframe belonging to particular user:
        preds = df_preds[df_preds['srch_id'] == item]
        # chunk of true dataframe belonging to particular user:
        real = df_actual[df_actual['srch_id'] == item]
        #for each occurence of user ID:
        
        for i in range(0,len(real[real['srch_id'] == item])):
            
            score = 0
            #property id from a row from submission df
            prop_pred = preds.iloc[i][1]
            
            #Get score for current prop_id
            if (real[real['prop_id'] == prop_pred]['booking_bool'] == 1).iloc[0]:
                score = 5
            elif (real[real['prop_id'] == prop_pred]['click_bool'] == 1).iloc[0]:
                score = 1
            else:
                score = 0
            
            #apply NDCG to score
            DCG_score = score/(np.log2((i+1)+1))
            IDCG_per_id[score_index] += DCG_score
            
            if i <= 4:
                #print(f"i is {i} and adding {DCG_score} to DCG_5")
                DCG_5_per_id[score_index] += DCG_score


    print(f"Total Time: {time.time() - start}")
    

    # Calculat Average NDCG_5 score over all unique srch_ids
    NDCG_5 = 0
    for i in range(0,len(ids)):
        IDCG = IDCG_per_id[i]
        DCG_5 = DCG_5_per_id[i]
        NDCG_5 += DCG_5/IDCG
        
    
    NDCG_5 = NDCG_5/len(ids)
        
    
    print(f"Average score (NDCG_5 score) is: {NDCG_5}")
    return(IDCG_per_id,DCG_5_per_id)